In [0]:
from pyspark.sql.functions import col, sum, avg, count, year, month, desc, when, lit
import matplotlib.pyplot as plt
import pandas as pd

# 1. Carregar tabelas gold com os nomes CORRETOS
casos_por_municipio = spark.table("gold_dengue_municipio")
casos_mensal = spark.table("gold_dengue_temporal")
dengue_demografico = spark.table("gold_dengue_demografico")
dengue_gravidade = spark.table("gold_dengue_gravidade")

# Amostra dos dados
print("=== gold_dengue_municipio ===")
casos_por_municipio.show(5, truncate=False)

print("=== gold_dengue_temporal ===")
casos_mensal.show(5, truncate=False)

print("=== gold_dengue_demografico ===")
dengue_demografico.show(5, truncate=False)

print("=== gold_dengue_gravidade ===")
dengue_gravidade.show(5, truncate=False)

# 1. Total de casos por estado (agrupando por UF)
# Como a tabela não tem a sigla da UF separada, vamos agrupar apenas pelos municípios com mais casos
top_municipios = casos_por_municipio.orderBy(desc("total_casos")).limit(10)
print("\n--- Top 10 Municípios com mais casos ---")
top_municipios.show()

# 2. Evolução mensal dos casos (últimos 12 meses)
casos_mensal_recente = casos_mensal \
    .orderBy(col("ano").desc(), col("mes").desc()) \
    .limit(12)
print("\n--- Evolução Mensal ---")
casos_mensal_recente.show()

# 3. Distribuição por Raça/Cor (demográfico)
raca_cor = dengue_demografico.groupBy("cs_raca").agg(
    sum("total_casos").alias("total")
).orderBy(desc("total"))
print("\n--- Casos por Raça/Cor ---")
raca_cor.show()

# 4. Casos por Gravidade
gravidade_total = dengue_gravidade.groupBy("classi_fin").agg(
    sum("total_casos").alias("total")
).orderBy(desc("total"))
print("\n--- Casos por Gravidade ---")
gravidade_total.show()

# 5. Gerar gráfico simples com pandas (Top 10 Municípios)
##df_pd = top_municipios.toPandas()
##plt.figure(figsize=(12,6))
##plt.bar(df_pd["municipio"], df_pd["total_casos"], color="coral")
##plt.title("Top 10 Municípios com mais casos de dengue")
##plt.xlabel("Município")
##plt.ylabel("Total de Casos")
##plt.xticks(rotation=45, ha='right')
##plt.tight_layout()
##plt.show()